In [1]:
"""
BUILD MATCHED bat_score=0 COMPARISON SAMPLE
===============================================
Goal: test whether the ~72% zero-comment rate found for bat_score>0
posts is elevated relative to baseline, or just how these subreddits
behave generally.

Approach: draw a sample of bat_score=0 posts, STRATIFIED to match the
subreddit x year distribution of the bat_score>0 population. A random
sample of bat_score=0 posts wouldn't be a fair comparison if, say, the
bat_score>0 posts skew toward r/ciso and 2023-2024, while a random
sample skews toward r/sysadmin and 2019 — differences in engagement
could then reflect subreddit/era effects, not burnout content itself.

ASSUMPTIONS TO VERIFY:
    - bat_posts_results_with_dates_v2.csv confirmed to have columns:
      post_id, bat_score, subreddit, created_date, year (144,652 rows)

Reads:
    bat_posts_results_with_dates_v2.csv   (post_id, bat_score, subreddit,
                                            created_date, year — all in one file)

Writes:
    bat0_comparison_sample.csv
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATA_DIR = '/Users/nadia/Desktop/redditRun_june/comment_data/'
BAT_DATES_FILE =  '/Users/nadia/Desktop/redditRun_june/bat_posts_results_with_dates_v2.csv'
OUTPUT_DIR = DATA_DIR

RANDOM_STATE = 42


def load_and_merge():
    cols = ['post_id', 'bat_score', 'subreddit', 'created_date', 'year']
    merged = pd.read_csv(BAT_DATES_FILE, usecols=cols)
    merged['post_id'] = merged['post_id'].astype(str)
    print(f"Loaded {len(merged):,} posts from {BAT_DATES_FILE}")

    merged['created_date'] = pd.to_datetime(merged['created_date'], errors='coerce')
    # year column already present in the file, but recompute from created_date
    # in case of any mismatch between the two
    merged['year'] = merged['created_date'].dt.year

    n_missing_meta = merged['subreddit'].isna().sum() + merged['year'].isna().sum()
    if n_missing_meta > 0:
        print(f"⚠ {n_missing_meta:,} rows missing subreddit/year — dropping those")
        merged = merged.dropna(subset=['subreddit', 'year'])

    return merged


def build_stratified_sample(merged):
    positive = merged[merged['bat_score'] > 0]
    zero = merged[merged['bat_score'] == 0]

    print(f"\nbat_score>0 posts (target distribution source): {len(positive):,}")
    print(f"bat_score=0 posts (sampling pool): {len(zero):,}")

    # Target: for each (subreddit, year) cell, sample the SAME number of
    # bat_score=0 posts as there are bat_score>0 posts in that cell.
    target_counts = positive.groupby(['subreddit', 'year']).size()

    sampled_parts = []
    shortfall_cells = []

    for (sub, yr), n_target in target_counts.items():
        pool = zero[(zero['subreddit'] == sub) & (zero['year'] == yr)]
        n_available = len(pool)
        n_draw = min(n_target, n_available)
        if n_draw < n_target:
            shortfall_cells.append((sub, yr, n_target, n_available))
        if n_draw > 0:
            sampled_parts.append(pool.sample(n_draw, random_state=RANDOM_STATE))

    sample = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else pd.DataFrame()

    print(f"\nMatched comparison sample drawn: {len(sample):,} posts "
          f"(target was {target_counts.sum():,})")

    if shortfall_cells:
        print(f"\n⚠ {len(shortfall_cells)} (subreddit, year) cells had fewer bat_score=0 "
              f"posts available than needed:")
        for sub, yr, target, avail in shortfall_cells[:10]:
            print(f"    {sub} / {yr}: needed {target}, only {avail} available")
        if len(shortfall_cells) > 10:
            print(f"    ... and {len(shortfall_cells) - 10} more")

    return sample


if __name__ == '__main__':
    print("=" * 80)
    print("BUILD MATCHED bat_score=0 COMPARISON SAMPLE")
    print("=" * 80)

    merged = load_and_merge()
    sample = build_stratified_sample(merged)

    out_cols = ['post_id', 'subreddit', 'year', 'bat_score']
    sample[out_cols].to_csv(OUTPUT_DIR + 'bat0_comparison_sample.csv', index=False)
    print(f"\n✓ bat0_comparison_sample.csv saved  →  {len(sample):,} posts")
    print("\nNext: run the comment-count comparison script against both this file "
          "and bat_score_pos.csv to test whether zero-comment rates differ.")

BUILD MATCHED bat_score=0 COMPARISON SAMPLE
Loaded 144,652 posts from /Users/nadia/Desktop/redditRun_june/bat_posts_results_with_dates_v2.csv

bat_score>0 posts (target distribution source): 12,237
bat_score=0 posts (sampling pool): 132,415

Matched comparison sample drawn: 12,237 posts (target was 12,237)

✓ bat0_comparison_sample.csv saved  →  12,237 posts

Next: run the comment-count comparison script against both this file and bat_score_pos.csv to test whether zero-comment rates differ.
